# The O(√T) Regret Bound for Online Gradient Descent

Wiki reference for [the OGD regret bound](https://ml-viz-ruby.vercel.app/wiki/ogd-regret-bound).

**The idea in one sentence.** With step η = D/(G√T), online gradient descent's cumulative regret against the best fixed decision is at most D·G·√T — sublinear, so average regret vanishes.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — OGD with a projection

Feasible set `K = [-1, 1]` (diameter `D = 2`); losses `f_t(x) = |x - y_t|` with `y_t ∈ [-1,1]`, so subgradients are `sign(x - y_t)` with norm `G = 1`. OGD steps and projects back into `K`. The bound predicts regret `≤ D·G·√T`.

In [ ]:
def project(x, lo=-1.0, hi=1.0):
    return min(hi, max(lo, x))

def run_ogd(y, eta):
    x, loss, comp = 0.0, 0.0, 0.0
    x_star = float(np.clip(np.median(y), -1, 1))   # best fixed point in K (abs loss -> median)
    regret = np.empty(len(y))
    for t, yt in enumerate(y, start=1):
        loss += abs(x - yt)
        comp += abs(x_star - yt)
        regret[t - 1] = loss - comp
        g = np.sign(x - yt)             # subgradient of |x - y_t|
        x = project(x - eta * g)
    return regret

T = 20000
y = np.clip(np.random.randn(T) * 0.5, -1, 1)   # stationary, inside K
D, G = 2.0, 1.0
eta = D / (G * np.sqrt(T))
regret = run_ogd(y, eta)
bound = D * G * np.sqrt(T)
print(f'OGD regret R_T = {regret[-1]:.1f}   theoretical bound D·G·√T = {bound:.1f}')
print(f'average regret R_T/T = {regret[-1] / T:.5f}')

## 2. Validate — the empirical regret obeys the bound

The proof guarantees `regret ≤ D·G·√T` for `η = D/(G√T)`. We assert it holds, and that a learner playing a fixed poor point pays *linear* regret instead.

In [ ]:
assert 0 <= regret[-1] <= bound, 'empirical regret must be non-negative and under D·G·√T'

def run_fixed(y, x0):
    loss, comp = 0.0, 0.0
    x_star = float(np.clip(np.median(y), -1, 1))
    reg = np.empty(len(y))
    for t, yt in enumerate(y, start=1):
        loss += abs(x0 - yt); comp += abs(x_star - yt)
        reg[t - 1] = loss - comp
    return reg

regret_fixed = run_fixed(y, x0=1.0)   # a fixed, poor decision
assert regret_fixed[-1] > 5 * regret[-1], 'the fixed learner should pay far larger (linear) regret'
print('OGD obeys R_T ≤ D·G·√T; the fixed learner grows linearly ✓')

## 3. Visualize it — regret vs the √T envelope

Plot OGD's cumulative regret under the `D·G·√T` curve, with the fixed learner for contrast.

In [ ]:
ts = np.arange(1, T + 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts, regret_fixed, color='#f43f5e', label='fixed guess  (~T)')
ax.plot(ts, regret, color='#14b8a6', label='OGD  (~√T)')
ax.plot(ts, D * G * np.sqrt(ts), '--', color='#e2e8f0', label='bound D·G·√t')
ax.set_xlabel('rounds T'); ax.set_ylabel('cumulative regret')
ax.set_title('OGD regret stays under the D·G·√T envelope', color='white')
ax.legend(); ax.grid(alpha=0.2); plt.show()

**What to notice:** OGD's teal curve stays beneath the dashed `D·G·√t` envelope for all `t` — the proof's guarantee, verified empirically. Because the envelope grows like `√t`, average regret `R_t/t` decays to zero; the fixed learner's straight line never does.

## 4. Gotchas

- **Regret is vs the best FIXED action**, not a moving target; against a drifting comparator regret can even go negative (OGD tracks the shift). Use *dynamic* regret for tracking problems.
- **Step size is the knob.** `η = D/(G√T)` needs `T`; `η_t = D/(G√t)` removes that at a `3/2` constant. Strong convexity buys `O(log T)`.
- **Bounded gradients matter** — the `G` in the bound is real; clip or normalise features so no single example dominates.

## 5. Your turn

### Exercise — the tuned bound and average regret

Given `D`, `G`, `T`, return a tuple `(eta_star, regret_bound, avg_regret_bound)` where `eta_star = D/(G√T)`, `regret_bound = D·G·√T`, and the average is the bound divided by `T`.

In [ ]:
def ogd_bound(D, G, T):
    # TODO(you): return (eta_star, regret_bound, avg_regret_bound)
    return ...


In [ ]:
# Checks — run me
eta_star, rb, avg = ogd_bound(2.0, 1.0, 10000)
assert abs(eta_star - 0.02) < 1e-9, 'eta* = D/(G√T) = 2/(1·100) = 0.02'
assert abs(rb - 200.0) < 1e-9, 'bound = D·G·√T = 200'
assert abs(avg - 0.02) < 1e-9, 'average regret bound = 200/10000 = 0.02'
# average regret vanishes as T grows
assert ogd_bound(2, 1, 1_000_000)[2] < ogd_bound(2, 1, 10_000)[2]
print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def ogd_bound(D, G, T):
    eta_star = D / (G * np.sqrt(T))
    regret_bound = D * G * np.sqrt(T)
    return eta_star, regret_bound, regret_bound / T
```

</details>

## 6. Key takeaways

- Three steps: **projection inequality → convexity → telescoping** give `regret ≤ D²/(2η) + ηG²T/2`.
- Tuning `η = D/(G√T)` yields **`regret ≤ D·G·√T = O(√T)`**; average regret → 0.
- Strong convexity improves this to `O(log T)`; online-to-batch turns it into `O(1/√T)` risk.
- Back to [Online Learning & Regret](https://ml-viz-ruby.vercel.app/courses/streaming-ml/03-online-learning).